Research Question 3: How does the time between winning possession in your own half and the first shot correlate with the probability of scoring (based on xGoals)?

Retrieving the Data:

In [ ]:
import pandas as pd
import json
import matplotlib.pyplot as plt
import numpy as np

import soccerdata as sd


data = sd.WhoScored(leagues="GER-Bundesliga", seasons="2024/2025", headless=True)

# Creating a list of all the game IDs of the Bundesliga season 2024/2025
game_ids = data.read_schedule()["game_id"].to_list()

events = data.read_events(match_id=game_ids)

# Defining possession IDs for each possession
events["poss_change"] = events["team_id"] != events["team_id"].shift()
events["possession_id"] = events["poss_change"].cumsum()

shot_types = ["Goal", "MissedShots", "SavedShot", "ShotOnPost"]
events["is_shot"] = events["type"].isin(shot_types)

possessions_with_shot = events.groupby("possession_id")["is_shot"].transform("any")

shot_results = events[events["is_shot"]].groupby("possession_id")["is_goal"].any()

possession_win = ["Tackle", "Interception", "BallRecovery"]

# Determining the time difference between possession gain and shot
events["poss_gain_time"] = (events["minute"] * 60) + events["second"]
first_shot_time = events[events["is_shot"]].groupby("possession_id")["poss_gain_time"].min()

# Filter for all events that were a possession gain in the own half, which resulted in a shot
own_half_wins_with_shot = events[
    (events["type"].isin(possession_win)) & 
    (events["outcome_type"] == "Successful") & 
    (events["x"] < 50) &
    (events["poss_change"] == True) &
    (possessions_with_shot)
].copy()

own_half_wins_with_shot = own_half_wins_with_shot.sort_values("poss_gain_time").drop_duplicates("possession_id", keep="first")
# Merge the first_shot_time table
own_half_wins_with_shot["shot_time"] = own_half_wins_with_shot["possession_id"].map(first_shot_time)

own_half_wins_with_shot["time_delta"] = own_half_wins_with_shot["shot_time"] - own_half_wins_with_shot["poss_gain_time"]

# Merge the shot_result table
own_half_wins_with_shot["is_goal"] = own_half_wins_with_shot["possession_id"].map(shot_results)

# Make a dataframe with only the relevant data
df_RQ3 = own_half_wins_with_shot[["is_goal","time_delta"]].copy()

df_RQ3 = df_RQ3.sort_values(by="time_delta")

df_RQ3.to_json("RQ3.json", orient="records", indent=4)
df_RQ3.to_csv("RQ3.csv", index=False)


Creating the Visualization:

In [ ]:
# Defining intervalls
bins = [0, 10, 15, 20, 25, 30, 35, 40, 45, 50, float("inf")]
labels = ["0-10s", "10-15s", "15-20s", "20-25s", "25-30s", "30-35s", "35-40s", "40-45s", "45-50s", "50s+"]

# Assigning intervalls to each element of the dataframe
df_RQ3["intervalls"] = pd.cut(
    df_RQ3["time_delta"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

goals = []
no_goals = []

for label in labels:
    subset = df_RQ3[df_RQ3["intervalls"] == label]

    g = int(subset[subset["is_goal"]==True].shape[0])
    ng = int(subset[subset["is_goal"] == False].shape[0])

    goals.append(g)
    no_goals.append(ng)

plt.figure(figsize=(14, 8))

ax = plt.gca()
ax.set_axisbelow(True)

plt.grid(axis="y")

plt.bar(labels, goals, bottom=no_goals, color="#27ae60", label="Goal", edgecolor="white")
plt.bar(labels, no_goals, color="#FF0000", label="No Goal", edgecolor="white")

plt.xticks(fontsize=18)
plt.yticks(fontsize=18)

plt.legend(["Goal", "No Goal"], fontsize=30)

plt.tight_layout()

plt.savefig("GoalsBarRQ3.png")

plt.show()

In [ ]:
percentages = []
for x in range(len(goals)):
    percentages.append(round(goals[x]/(no_goals[x]+goals[x])*100))

plt.figure(figsize=(14, 8))

plt.plot(labels, percentages, color="#27ae60", marker='.', linewidth=2, markersize=8)

ax = plt.gca()
ax.set_axisbelow(True)
plt.grid(axis="y", linestyle='-', alpha=0.7)

plt.title("Goal Conversion Rate per Time Interval", fontsize=24)
plt.xlabel("Seconds after possession win", fontsize=16)
plt.ylabel("Conversion Rate (%)", fontsize=16)

plt.savefig("GoalsLineRQ3.png")

plt.show()